In [ ]:
import os
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY from environment

SYSTEM_PROMPT = """
You are an insurance support assistant for general educational guidance.
- Do NOT give legal/financial advice.
- Ask 1 clarifying question if needed.
- Give practical next steps and a short checklist.
"""

def ask_insurance(question: str, model: str = "gpt-4.1-mini") -> str:
    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
    )
    return resp.output_text

print(ask_insurance("Do I need renters insurance if my landlord has insurance?"))

In [ ]:

PROMPTS = {
    "simple": SYSTEM_PROMPT,
    "extra_cautious": """
You are an insurance assistant for general guidance only.
- Always include a disclaimer: not legal/financial advice.
- Provide assumptions if info is missing.
- End with a 3-bullet checklist.
""",
    "super_structured": """
You are an insurance educator.
Format:
1) Summary
2) Key factors
3) Common pitfalls
4) Next steps (checklist)
No legal/financial advice.
"""
}

QUESTIONS = [
    "What’s the difference between collision and comprehensive?",
    "My car was stolen—what should I do first?",
    "Can you estimate how much car insurance will cost?"
]

def score(text: str) -> int:
    t = text.lower()
    points = 0
    points += 1 if "checklist" in t or "next step" in t else 0
    points += 1 if "not legal" in t or "not financial" in t else 0
    points += 1 if "-" in text or "•" in text else 0
    return points

results = {}
for name, sp in PROMPTS.items():
    total = 0
    for q in QUESTIONS:
        ans = client.responses.create(
            model="gpt-4.1-mini",
            input=[{"role":"system","content":sp},{"role":"user","content":q}]
        ).output_text
        total += score(ans)
    results[name] = total

best = max(results, key=results.get)
best, results